In [0]:
from pyspark.sql import functions as F

TARGET_TABLE = "finance_intelligence_hub.bronze.stocks_info"

# Add the column if it doesn't already exist
existing_cols = {f.name for f in spark.table(TARGET_TABLE).schema.fields}

if "dwh_loaded_at" not in existing_cols:
    spark.sql(f"ALTER TABLE {TARGET_TABLE} ADD COLUMNS (dwh_loaded_at TIMESTAMP)")
    print("Added dwh_loaded_at column.")
else:
    print("dwh_loaded_at already exists, skipping ALTER.")

# Backfill all existing rows with the fixed date
spark.sql(f"""
    UPDATE {TARGET_TABLE}
    SET dwh_loaded_at = TIMESTAMP('2026-07-06 00:00:00')
    WHERE dwh_loaded_at IS NULL
""")

backfilled = spark.sql(
    f"SELECT COUNT(*) FROM {TARGET_TABLE} WHERE dwh_loaded_at = TIMESTAMP('2026-07-06 00:00:00')"
).first()[0]

print(f"Backfilled {backfilled} rows with dwh_loaded_at = 2026-07-06.")